# PPO car trainer - many maps (GPU)
Trains on hundreds of random procedural tracks plus `TRAIN_REAL_MAPS`, so the car learns to drive **unseen** roads. `TEST_MAPS` are never trained on; they are evaluated every `EVAL_EVERY` updates (greedy, no crashes = success). `multi_best.pt/.npz` is chosen on separate random validation tracks, not on the test maps. Put the map `.json` files next to this notebook, then **Run All**. Outputs in `runs_multi/` (all files start with `multi_`). To stop: Interrupt kernel (saves `multi_final.pt/.npz` and runs a last evaluation).

In [ ]:
import json, math, os, time, collections
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

# ════════════════════════════ CONFIG ════════════════════════════
# --- maps -----------------------------------------------------------------
TRAIN_REAL_MAPS = ["first.json"]                          # hand-drawn maps used for training
TEST_MAPS       = ["circle.json", "second.json", "training.json"]   # NEVER trained on, never used to pick checkpoints
N_GEN_TRAIN     = 400      # random procedural tracks to train on
N_GEN_VAL       = 24       # separate random tracks, used to choose best.pt
REAL_FRAC       = 0.15     # share of episodes that start on a real training map
TRACK_SEED      = 1234

OUT_DIR      = "runs_multi"
RESUME       = None        # e.g. "runs/final.pt" (a single-map model) to warm-start
SEED         = 0

# --- run ----------------------------------------------------------------------
NUM_ENVS     = 256         # cars simulated in parallel
ROLLOUT      = 128         # decisions per car between PPO updates (batch = 32768)
TOTAL_STEPS  = 500_000_000
MAX_HOURS    = None        # wall-clock budget, e.g. 11.5 -> saves and exits cleanly
LOG_EVERY    = 10          # updates between printed log lines
SAVE_EVERY   = 25          # updates between overwriting latest.pt
SNAP_EVERY   = 1000        # updates between permanent snap_XXXXX copies
EVAL_EVERY   = 250         # updates between evaluations (test maps + validation tracks)
EVAL_STEPS   = 1200        # decisions per evaluation episode (20 s)
EVAL_CARS_TEST = 16        # cars per test map (half drive each direction)

# --- PPO ----------------------------------------------------------------------
GAMMA, LAM, CLIP_EPS = 0.995, 0.95, 0.2
LR_ACTOR, LR_CRITIC  = 3e-4, 1e-3
ENT_COEF, ENT_DECAY, ENT_MIN = 0.05, 0.9985, 0.01
EPOCHS, MINIBATCH    = 4, 4096
TARGET_KL, MAX_GRAD  = 0.03, 0.5
HIDDEN               = 128

# --- simulation (identical to map_reader.py, AI mode) -------------------------
FPS, SUBSTEPS      = 60, 6
FRAME_DT           = 1.0 / FPS
SUB_DT             = FRAME_DT / SUBSTEPS
WHEELBASE          = 28.0
MAX_SPEED          = 240.0
ACCEL, BRAKE_F     = 280.0, 280.0
DRAG_K             = 1.6
MAX_STEER_DEG      = 28.0
MAX_STEER          = math.radians(MAX_STEER_DEG)
STEER_RATE, STEER_RTN = math.radians(90), math.radians(150)
RAY_MAX, RAY_N     = 220.0, 10
RAY_FOV            = 160.0
MARGIN             = 6.0
STALL_TIMEOUT, STALL_EPS = 5.0, 4.0
CRASH_PENALTY      = -15.0
MAX_EP_STEPS       = 1800

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OBS_DIM, ACT_DIM = 2 * RAY_N + 2, 9
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(no GPU found!)")


In [ ]:
# ═════════════════ MAPS: real files + procedural tracks ═════════════════
def build_edges(pts, half_w):
    """Same offsetting rule as environment.py (perpendicular from neighbouring points); pts is a closed loop."""
    d = np.roll(pts, -1, 0) - np.roll(pts, 1, 0)
    p = np.stack([-d[:, 1], d[:, 0]], 1) / np.maximum(np.hypot(d[:, 0], d[:, 1]), 1e-6)[:, None]
    close = lambda a: np.vstack([a, a[:1]])
    return close(pts + p * half_w), close(pts - p * half_w)


def load_real(path):
    with open(path) as f:
        m = json.load(f)
    a = lambda k: np.array(m[k], dtype=np.float32)
    return dict(name=os.path.basename(path), center=a("center"), left=a("left"), right=a("right"),
                half_w=float(m.get("lane_width", 60)) / 2)


def _catmull_closed(P, samples=30):
    out, n = [], len(P)
    t = np.linspace(0, 1, samples, endpoint=False)[:, None]
    for i in range(n):
        p0, p1, p2, p3 = P[(i - 1) % n], P[i], P[(i + 1) % n], P[(i + 2) % n]
        out.append(0.5 * (2 * p1 + (-p0 + p2) * t + (2 * p0 - 5 * p1 + 4 * p2 - p3) * t ** 2
                          + (-p0 + 3 * p1 - 3 * p2 + p3) * t ** 3))
    return np.concatenate(out)


def _resample(curve, next_step, mean_step):
    """Walk around a closed curve placing points `next_step()` px apart (arc length)."""
    closed = np.vstack([curve, curve[:1]])
    s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(closed, axis=0), axis=1))])
    pos = [0.0]
    while pos[-1] + next_step() < s[-1] - 0.5 * mean_step:
        pos.append(pos[-1] + next_step.last)
    pos = np.array(pos)
    return np.stack([np.interp(pos, s, closed[:, 0]), np.interp(pos, s, closed[:, 1])], 1)


class _Step:
    def __init__(self, fn): self.fn, self.last = fn, 0.0
    def __call__(self): self.last = self.fn(); return self.last


def _min_radius(q, k=6):
    a, b, c = np.roll(q, k, 0), q, np.roll(q, -k, 0)
    cross = np.abs((b - a)[:, 0] * (c - a)[:, 1] - (b - a)[:, 1] * (c - a)[:, 0])
    lens = np.linalg.norm(b - a, axis=1) * np.linalg.norm(c - b, axis=1) * np.linalg.norm(c - a, axis=1)
    return float((lens / (2 * np.maximum(cross, 1e-9))).min())


def _separated(q, step, min_dist=72.0, min_arc=150.0):
    n = len(q)
    D = np.linalg.norm(q[:, None] - q[None], axis=-1)
    i = np.arange(n); arc = np.minimum(np.abs(i[:, None] - i[None]), n - np.abs(i[:, None] - i[None])) * step
    return bool(D[arc > min_arc].min() >= min_dist)


def _chaikin_closed(P, it):
    for _ in range(it):
        Q = np.roll(P, -1, 0)
        P = np.stack([0.75 * P + 0.25 * Q, 0.25 * P + 0.75 * Q], 1).reshape(-1, 2)
    return P


def _radial_closed(rng, samples=600):
    """Polar curve r(theta) = 1 + sum of random low harmonics: gives lobes, bays and necks."""
    th = np.linspace(0, 2 * np.pi, samples, endpoint=False)
    r = np.ones(samples)
    for k in range(2, int(rng.integers(4, 8))):
        r += rng.uniform(0, 0.6 / k ** 0.5) * np.cos(k * th + rng.uniform(0, 2 * np.pi))
    r = np.maximum(r, 0.2)
    return np.stack([r * np.cos(th), r * np.sin(th)], 1)


def _smooth_closed(curve, sigma):
    """Circular Gaussian smoothing along the curve (rounds sharp tips; the track is rescaled afterwards)."""
    n = len(curve); k = np.arange(-3 * int(sigma) - 1, 3 * int(sigma) + 2)
    w = np.exp(-k ** 2 / (2 * sigma ** 2)); w /= w.sum()
    idx = (np.arange(n)[:, None] + k[None]) % n
    return (curve[idx] * w[None, :, None]).sum(1)


def _notched_closed(rng, samples=600):
    """Ring with 1-2 deep notches in its outline -> horseshoes (one notch) and dumbbells (two)."""
    th = np.linspace(0, 2 * np.pi, samples, endpoint=False)
    r = np.ones(samples)
    for k in range(2, 4):
        r += rng.uniform(0, 0.12) * np.cos(k * th + rng.uniform(0, 2 * np.pi))
    t0 = rng.uniform(0, 2 * np.pi)
    centres = [t0] if rng.random() < 0.5 else [t0, t0 + np.pi + rng.uniform(-0.4, 0.4)]
    for c in centres:
        d = np.angle(np.exp(1j * (th - c)))
        r -= rng.uniform(0.45, 0.85) * np.exp(-d ** 2 / (2 * rng.uniform(0.25, 0.6) ** 2))
    r = np.maximum(r, 0.15)
    return _smooth_closed(np.stack([r * np.cos(th), r * np.sin(th)], 1), rng.uniform(6, 24))


def gen_track(rng, half_w=30.0):
    """Random closed track in one of three styles (smooth spline ring, polygon with rounded corners,
    wobbly radial profile), including concave bays, necks and hairpins; then a hand-drawn-like polyline."""
    for _ in range(5000):
        style = int(rng.integers(0, 4))
        if style == 2:
            dense = _radial_closed(rng)
        elif style == 3:
            dense = _notched_closed(rng)
        else:
            n = int(rng.integers(5, 15))
            ang = np.sort((np.arange(n) + rng.uniform(-0.4, 0.4, n)) * 2 * np.pi / n)
            rad = 1.0 - rng.uniform(0.15, 0.8) * rng.random(n)
            P = np.stack([np.cos(ang) * rad, np.sin(ang) * rad], 1)
            dense = _catmull_closed(P) if style == 0 else _chaikin_closed(P, int(rng.integers(2, 5)))
        dense = dense * np.array([1.0, rng.uniform(0.45, 1.0)])
        per = np.linalg.norm(np.diff(np.vstack([dense, dense[:1]]), axis=0), axis=1).sum()
        dense = dense * (rng.uniform(800, 2000) / per)
        if rng.random() < 0.5:
            dense[:, 0] *= -1
        even = _resample(dense, _Step(lambda: 4.0), 4.0)
        if _min_radius(even) < rng.uniform(30, 60) or not _separated(even, 4.0, min_dist=68.0):
            continue
        mean_ds = rng.uniform(18, 45)                               # irregular spacing like mouse-drawn maps
        pts = _resample(dense, _Step(lambda: mean_ds * rng.uniform(0.5, 1.5)), mean_ds)
        pts = (pts + rng.normal(0, 1.5, pts.shape)).astype(np.float32)
        left, right = build_edges(pts, half_w)
        return dict(name="gen", center=np.vstack([pts, pts[:1]]), left=left.astype(np.float32),
                    right=right.astype(np.float32), half_w=half_w, style=style)
    raise RuntimeError("could not generate a valid track")


class MapPool:
    """Many maps padded into tensors, so every car can be on a different map."""
    def __init__(self, maps, weights=None):
        self.names = [m["name"] for m in maps]
        M = self.M = len(maps)
        S = max(2 * (len(m["left"]) - 1) for m in maps)
        C = max(len(m["center"]) - 1 for m in maps)
        far = 1e6                                                    # padding = segments that are never hit
        WA = np.full((M, S, 2), far, np.float32); WB = WA.copy()
        CA = np.full((M, C, 2), far, np.float32); CB = CA.copy()
        ncs = np.zeros(M, np.int64)
        for k, m in enumerate(maps):
            a = np.concatenate([m["left"][:-1], m["right"][:-1]]); b = np.concatenate([m["left"][1:], m["right"][1:]])
            WA[k, :len(a)], WB[k, :len(a)] = a, b
            c = m["center"]; CA[k, :len(c) - 1], CB[k, :len(c) - 1] = c[:-1], c[1:]; ncs[k] = len(c) - 1
        t = lambda x: torch.tensor(x, device=DEVICE)
        self.WA, self.WE = t(WA), t(WB - WA)
        self.WL2 = (self.WE ** 2).sum(-1).clamp_min(1e-9)
        self.CA, self.CE = t(CA), t(CB - CA)
        self.CL = self.CE.norm(dim=-1).clamp_min(1e-6)
        self.CT = self.CE / self.CL[..., None]
        self.NCS = t(ncs)
        self.LEN = t(np.array([float(np.linalg.norm(np.diff(m['center'], axis=0), axis=1).sum()) for m in maps], np.float32))
        self.HW = t(np.array([m["half_w"] for m in maps], np.float32))
        w = np.ones(M) if weights is None else np.asarray(weights, dtype=np.float64)
        self.weights = t((w / w.sum()).astype(np.float32))


def build_pools():
    both = set(TRAIN_REAL_MAPS) & set(TEST_MAPS)
    if both:
        raise ValueError(f"{sorted(both)} is in both TRAIN_REAL_MAPS and TEST_MAPS - a test map must never be trained on")
    rng = np.random.default_rng(TRACK_SEED)
    gen_train = [gen_track(rng) for _ in range(N_GEN_TRAIN)]
    gen_val = [gen_track(np.random.default_rng(TRACK_SEED + 1 + i)) for i in range(N_GEN_VAL)]
    real_train = [load_real(p) for p in TRAIN_REAL_MAPS]
    test = [load_real(p) for p in TEST_MAPS]
    w = [(1 - REAL_FRAC) / max(len(gen_train), 1)] * len(gen_train) + \
        [REAL_FRAC / max(len(real_train), 1)] * len(real_train)
    train_pool = MapPool(gen_train + real_train, w)
    perim = [float(np.linalg.norm(np.diff(m["center"], axis=0), axis=1).sum()) for m in gen_train]
    print(f"train pool: {len(gen_train)} generated tracks (length {min(perim):.0f}-{max(perim):.0f} px) "
          f"+ real {TRAIN_REAL_MAPS}; validation: {len(gen_val)} generated; test: {TEST_MAPS}")
    for m in real_train + test:                                     # warn about pinched spots (unpassable by anyone)
        c = m["center"]
        P = np.concatenate([a + np.linspace(0, 1, 10, endpoint=False)[:, None] * (b - a) for a, b in zip(c[:-1], c[1:])])
        A = np.concatenate([m["left"][:-1], m["right"][:-1]]); B = np.concatenate([m["left"][1:], m["right"][1:]])
        AB = B - A; u = np.clip(((P[:, None] - A[None]) * AB[None]).sum(-1) / np.maximum((AB ** 2).sum(-1), 1e-9)[None], 0, 1)
        d = np.linalg.norm(P[:, None] - (A[None] + u[..., None] * AB[None]), axis=-1).min(1)
        if (d < MARGIN).any():
            print(f"  WARNING {m['name']}: {100 * (d < MARGIN).mean():.1f}% of the centre line is within {MARGIN:.0f}px of a wall (unpassable spot)")
    return train_pool, MapPool(gen_val), MapPool(test)


TRAIN_POOL, VAL_POOL, TEST_POOL = build_pools()


In [ ]:
# ═════════════════════ BATCHED SIMULATOR (multi-map) ═════════════════════
_front = [math.radians(-RAY_FOV / 2 + i * RAY_FOV / (RAY_N - 1)) for i in range(RAY_N)]
_rear  = [math.radians(180 - RAY_FOV / 2 + i * RAY_FOV / (RAY_N - 1)) for i in range(RAY_N)]
REL_ANG = torch.tensor(_front + _rear, dtype=torch.float32, device=DEVICE)

# action table: COAST THROTTLE BRAKE LEFT RIGHT THR_L THR_R BRK_L BRK_R
ACT_THR = torch.tensor([0, 1, 0, .3, .3, .9, .9, 0, 0], dtype=torch.float32, device=DEVICE)
ACT_BRK = torch.tensor([0, 0, 1, 0, 0, 0, 0, 1, 1], dtype=torch.bool, device=DEVICE)
ACT_STR = torch.tensor([0, 0, 0, -1, 1, -1, 1, -1, 1], dtype=torch.float32, device=DEVICE)


def cast_rays(pos, heading, wa, we):
    """pos (E,2), heading (E,), wall segments wa/we (E,S,2) -> frac (E, 2*RAY_N)."""
    ang = heading[:, None] + REL_ANG[None, :]
    dx, dy = torch.cos(ang)[..., None], torch.sin(ang)[..., None]          # (E,R,1)
    aox = wa[..., 0][:, None, :] - pos[:, 0][:, None, None]                # (E,1,S)
    aoy = wa[..., 1][:, None, :] - pos[:, 1][:, None, None]
    ex, ey = we[..., 0][:, None, :], we[..., 1][:, None, :]
    denom = dx * ey - dy * ex                                              # (E,R,S)
    ok = denom.abs() > 1e-10
    denom = torch.where(ok, denom, torch.ones_like(denom))
    t = (aox * ey - aoy * ex) / denom
    u = (aox * dy - aoy * dx) / denom
    hit = ok & (t > 0) & (u >= 0) & (u <= 1)
    t = torch.where(hit, t, torch.full_like(t, RAY_MAX))
    return t.amin(-1).clamp(max=RAY_MAX) / RAY_MAX


def wall_dist(pos, wa, we, wl2):
    ap = pos[:, None, :] - wa
    s = ((ap * we).sum(-1) / wl2).clamp(0, 1)
    return (pos[:, None, :] - (wa + s[..., None] * we)).norm(dim=-1).amin(-1)


def track_state(pos, heading, ca, ce, cl, ct, dirn):
    """Distance from the centre line, and cos(angle between heading and the direction of travel along the road)."""
    ap = pos[:, None, :] - ca
    s = ((ap * ce).sum(-1) / cl ** 2).clamp(0, 1)
    d = (pos[:, None, :] - (ca + s[..., None] * ce)).norm(dim=-1)
    lat, j = d.min(1)
    tang = ct.gather(1, j[:, None, None].expand(-1, 1, 2))[:, 0]
    return lat, dirn * (torch.cos(heading) * tang[:, 0] + torch.sin(heading) * tang[:, 1])


def reward_fn(speed, lat_norm, fwd):
    """Progress along the road minus an off-centre penalty."""
    return 3.0 * speed * fwd / MAX_SPEED - 2.0 * lat_norm ** 2


class Cars:
    """n cars, each on its own map (sampled from `pool`, or fixed via fixed_ids), driving in a random or fixed direction."""
    def __init__(self, n, pool, fixed_ids=None, fixed_dirs=None):
        self.n, self.pool, self.fixed_ids, self.fixed_dirs = n, pool, fixed_ids, fixed_dirs
        z = lambda *s: torch.zeros(*s, device=DEVICE)
        self.pos, self.anchor = z(n, 2), z(n, 2)
        self.heading, self.speed, self.steer, self.stall, self.dirn = z(n), z(n), z(n), z(n), torch.ones(n, device=DEVICE)
        self.steps = torch.zeros(n, dtype=torch.long, device=DEVICE)
        self.mid = torch.zeros(n, dtype=torch.long, device=DEVICE)
        p, m = pool, self.mid
        self.wa, self.we, self.wl2 = p.WA[m].clone(), p.WE[m].clone(), p.WL2[m].clone()
        self.ca, self.ce, self.cl, self.ct = p.CA[m].clone(), p.CE[m].clone(), p.CL[m].clone(), p.CT[m].clone()
        self.hw = p.HW[m].clone()
        self.reset(torch.ones(n, dtype=torch.bool, device=DEVICE))

    def reset(self, mask):
        k = int(mask.sum())
        if k == 0:
            return
        p = self.pool
        ids = (torch.multinomial(p.weights, k, replacement=True) if self.fixed_ids is None
               else self.fixed_ids[mask])
        self.mid[mask] = ids
        self.dirn[mask] = (torch.randint(0, 2, (k,), device=DEVICE).float() * 2 - 1 if self.fixed_dirs is None
                           else self.fixed_dirs[mask])
        self.wa[mask], self.we[mask], self.wl2[mask] = p.WA[ids], p.WE[ids], p.WL2[ids]
        self.ca[mask], self.ce[mask], self.cl[mask], self.ct[mask] = p.CA[ids], p.CE[ids], p.CL[ids], p.CT[ids]
        self.hw[mask] = p.HW[ids]
        ncs = p.NCS[ids]
        i = (torch.rand(k, device=DEVICE) * ncs).long().minimum(ncs - 1)   # random spot on the road
        a, e = p.CA[ids, i], p.CE[ids, i]
        d = self.dirn[mask]
        self.pos[mask] = a + torch.rand(k, 1, device=DEVICE) * e
        self.heading[mask] = torch.atan2(d * e[:, 1], d * e[:, 0])
        self.speed[mask] = 0.25 * MAX_SPEED
        self.steer[mask] = 0.0
        self.anchor[mask] = self.pos[mask]
        self.stall[mask] = 0.0
        self.steps[mask] = 0

    def observe(self):
        frac = cast_rays(self.pos, self.heading, self.wa, self.we)
        sn = (self.speed / MAX_SPEED).clamp(-1, 1)
        an = (torch.rad2deg(self.steer) / MAX_STEER_DEG).clamp(-1, 1)
        return torch.cat([frac, sn[:, None], an[:, None]], 1)

    def _physics(self, thr, brk, sdir, dt):
        sf = (self.speed.abs() / 20.0).clamp(max=1)
        diff = sdir * MAX_STEER - self.steer
        rate = torch.where(sdir.abs() > 0.01, STEER_RATE * sf, STEER_RTN * sf.clamp(min=0.1))
        self.steer = self.steer + torch.sign(diff) * torch.minimum(diff.abs(), rate * dt)

        v = self.speed
        braked = torch.where(v > 0, (v - BRAKE_F * dt).clamp(min=0),
                             torch.where(v < 0, (v + BRAKE_F * dt).clamp(max=0), v))
        a = v + thr * ACCEL * dt
        drag = (DRAG_K * a * a / MAX_SPEED + 15.0) * dt
        free = torch.where(a > 0, (a - drag).clamp(min=0),
                           torch.where(a < 0, (a + drag).clamp(max=0), a))
        self.speed = torch.where(brk, braked, free).clamp(-0.35 * MAX_SPEED, MAX_SPEED)

        turning = (self.speed.abs() > 0.1) & (self.steer.abs() > 0.0005)
        omega = self.speed * torch.tan(self.steer) / WHEELBASE
        self.heading = self.heading + torch.where(turning, omega * dt, torch.zeros_like(omega))
        self.pos = self.pos + self.speed[:, None] * dt * torch.stack(
            [torch.cos(self.heading), torch.sin(self.heading)], 1)

    def step(self, action):
        """Hold `action` for one frame. Returns obs_next, obs_final, reward, crashed, truncated.
        obs_final is the observation before auto-reset (used to bootstrap time-limit cut-offs)."""
        thr, brk, sdir = ACT_THR[action], ACT_BRK[action], ACT_STR[action]
        crashed = torch.zeros(self.n, dtype=torch.bool, device=DEVICE)
        for _ in range(SUBSTEPS):
            self._physics(thr, brk, sdir, SUB_DT)
            crashed |= wall_dist(self.pos, self.wa, self.we, self.wl2) < MARGIN
            far = (self.pos - self.anchor).norm(dim=1) >= STALL_EPS
            self.anchor = torch.where(far[:, None], self.pos, self.anchor)
            self.stall = torch.where(far, torch.zeros_like(self.stall), self.stall + SUB_DT)
            stalled = self.stall >= STALL_TIMEOUT
            crashed |= stalled
            self.stall = torch.where(stalled, torch.zeros_like(self.stall), self.stall)

        self.steps += 1
        obs_final = self.observe()
        lat, fwd = track_state(self.pos, self.heading, self.ca, self.ce, self.cl, self.ct, self.dirn)
        reward = torch.where(crashed, torch.full_like(self.speed, CRASH_PENALTY),
                             reward_fn(self.speed, lat / self.hw, fwd))
        trunc = (self.steps >= MAX_EP_STEPS) & ~crashed
        done = crashed | trunc
        if done.any():
            self.reset(done)
            obs_next = self.observe()
        else:
            obs_next = obs_final
        return obs_next, obs_final, reward, crashed, trunc


In [ ]:
# ═══════════════════════ PPO (PyTorch) ═══════════════════════
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        def mlp(out, gain):
            net = nn.Sequential(nn.Linear(OBS_DIM, HIDDEN), nn.Tanh(),
                                nn.Linear(HIDDEN, HIDDEN), nn.Tanh(), nn.Linear(HIDDEN, out))
            for i in (0, 2):
                nn.init.orthogonal_(net[i].weight, math.sqrt(2)); nn.init.zeros_(net[i].bias)
            nn.init.orthogonal_(net[4].weight, gain); nn.init.zeros_(net[4].bias)
            return net
        self.actor, self.critic = mlp(ACT_DIM, 0.01), mlp(1, 1.0)

    def forward(self, x):
        return self.actor(x), self.critic(x).squeeze(-1)   # logits, normalised value


class RunningNorm:
    """Running mean/std of returns; the critic is trained on normalised returns."""
    def __init__(self):
        self.mean, self.var, self.count = 0.0, 1.0, 1e-4

    def update(self, x):
        bm, bv, bc = x.mean().item(), x.var(unbiased=False).item(), x.numel()
        d, tot = bm - self.mean, self.count + bc
        self.var = (self.var * self.count + bv * bc + d * d * self.count * bc / tot) / tot
        self.mean += d * bc / tot
        self.count = tot

    @property
    def std(self): return math.sqrt(max(self.var, 1e-8))
    def norm(self, x): return (x - self.mean) / self.std
    def denorm(self, x): return x * self.std + self.mean


def gae(rew, val, done, last_val):
    """rew/val/done: (T,E). Returns advantages and returns, both (T,E)."""
    T = rew.shape[0]
    adv = torch.zeros_like(rew)
    last, nxt = torch.zeros_like(last_val), last_val
    for t in reversed(range(T)):
        mask = 1.0 - done[t]
        delta = rew[t] + GAMMA * nxt * mask - val[t]
        last = delta + GAMMA * LAM * mask * last
        adv[t], nxt = last, val[t]
    return adv, adv + val


def _atomic_save(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)


def export_npz(net, rn, path, info):
    """Weights in the NumPy format used by model.py (PPOAgent.load_npz) -> works with test_model.py."""
    a, c = net.actor, net.critic
    w = lambda l: l.weight.detach().cpu().numpy().T.astype(np.float64)
    b = lambda l: l.bias.detach().cpu().numpy().astype(np.float64)
    np.savez_compressed(
        path,
        a_W1=w(a[0]), a_b1=b(a[0]), a_W2=w(a[2]), a_b2=b(a[2]), a_W3=w(a[4]), a_b3=b(a[4]),
        c_W1=w(c[0]), c_b1=b(c[0]), c_W2=w(c[2]), c_b2=b(c[2]), c_W3=w(c[4]), c_b3=b(c[4]),
        episode=np.array(info["episodes"]), total_steps=np.array(info["env_steps"]),
        best_reward=np.array(info["best"]), entropy_coef=np.array(info["ent_coef"]),
        reward_history=np.array(info["hist"], dtype=np.float32),
        last_actor_loss=np.array(0.0), last_critic_loss=np.array(0.0),
        ret_norm_mean=np.array(rn.mean), ret_norm_var=np.array(rn.var), ret_norm_count=np.array(rn.count),
        max_speed=np.array(MAX_SPEED), max_steer_deg=np.array(MAX_STEER_DEG))


In [ ]:
# ═══════════════════════ EVALUATION + TRAIN ═══════════════════════
@torch.no_grad()
def evaluate(net, pool, per_map):
    """Greedy driving, `per_map` cars per map (half each direction). Returns per-map dict of arrays."""
    ids = torch.arange(pool.M, device=DEVICE).repeat_interleave(per_map)
    dirs = torch.where(torch.arange(len(ids), device=DEVICE) % 2 == 0, 1.0, -1.0)
    cars = Cars(len(ids), pool, fixed_ids=ids, fixed_dirs=dirs)
    obs = cars.observe()
    alive = torch.ones(len(ids), dtype=torch.bool, device=DEVICE)
    t_alive = torch.zeros(len(ids), device=DEVICE); speed_sum = torch.zeros(len(ids), device=DEVICE)
    dist = torch.zeros(len(ids), device=DEVICE)
    for _ in range(EVAL_STEPS):
        obs, _, _, crashed, _ = cars.step(net(obs)[0].argmax(-1))
        t_alive += alive.float(); speed_sum += cars.speed * alive.float()
        dist += cars.speed.clamp(min=0) * FRAME_DT * alive.float()
        alive &= ~crashed
    r = lambda x: x.reshape(pool.M, per_map)
    return dict(success=r(alive.float()).mean(1).cpu().numpy() * 100,
                time=(r(t_alive).mean(1) / FPS).cpu().numpy(),
                speed=(r(speed_sum).sum(1) / r(t_alive).sum(1).clamp_min(1)).cpu().numpy(),
                laps=(r(dist).mean(1) / pool.LEN).cpu().numpy())


def run_eval(net, update, env_steps):
    net.eval()
    test, val = evaluate(net, TEST_POOL, EVAL_CARS_TEST), evaluate(net, VAL_POOL, 2)
    net.train()
    print(f"  ── eval @ update {update} ({env_steps/1e6:.1f}M steps)  [greedy, {EVAL_STEPS/FPS:.0f}s; success = no crash; laps = distance driven / track length] ──")
    for k, name in enumerate(TEST_POOL.names):
        print(f"     TEST {name:<14} success {test['success'][k]:4.0f}%   laps {test['laps'][k]:4.1f}   alive {test['time'][k]:5.1f}s   speed {test['speed'][k]:4.0f}")
    print(f"     VAL  {len(VAL_POOL.names)} random tracks  success {val['success'].mean():4.0f}%   laps {val['laps'].mean():4.1f}   "
          f"alive {val['time'].mean():5.1f}s   speed {val['speed'].mean():4.0f}   (worst track {val['success'].min():.0f}%)")
    return float(val["success"].mean()) + 10 * float(val["speed"].mean()) / MAX_SPEED   # picks best.pt (never uses test maps)


def train():
    os.makedirs(OUT_DIR, exist_ok=True)
    torch.manual_seed(SEED); np.random.seed(SEED)
    E, T = NUM_ENVS, ROLLOUT
    net = ActorCritic().to(DEVICE)
    opt = torch.optim.Adam([{"params": net.actor.parameters(),  "lr": LR_ACTOR},
                            {"params": net.critic.parameters(), "lr": LR_CRITIC}], eps=1e-5)
    rn, ent_coef = RunningNorm(), ENT_COEF
    update, env_steps, episodes, best = 0, 0, 0, -float("inf")
    hist = collections.deque(maxlen=200)

    if RESUME:
        ck = torch.load(RESUME, map_location=DEVICE)
        net.load_state_dict(ck["net"]); opt.load_state_dict(ck["opt"])
        rn.mean, rn.var, rn.count = ck["rn"]
        ent_coef = max(ck["ent_coef"], 0.03)       # keep some exploration for the harder multi-map task
        print(f"warm-started from {RESUME}; best score / history reset")

    def snapshot(name, npz=False):
        info = dict(episodes=episodes, env_steps=env_steps, best=best, ent_coef=ent_coef, hist=list(hist))
        _atomic_save(dict(net=net.state_dict(), opt=opt.state_dict(), rn=(rn.mean, rn.var, rn.count),
                          ent_coef=ent_coef, update=update, env_steps=env_steps, episodes=episodes,
                          best=best, hist=list(hist)), os.path.join(OUT_DIR, name + ".pt"))
        if npz:
            export_npz(net, rn, os.path.join(OUT_DIR, name + ".npz"), info)

    cars = Cars(E, TRAIN_POOL)
    obs = cars.observe()
    ep_ret = torch.zeros(E, device=DEVICE); ep_len = torch.zeros(E, device=DEVICE)
    recent_ret, recent_len, recent_crash = (collections.deque(maxlen=200) for _ in range(3))

    B_obs = torch.zeros(T, E, OBS_DIM, device=DEVICE)
    B_act = torch.zeros(T, E, dtype=torch.long, device=DEVICE)
    B_lp, B_val, B_rew, B_done = (torch.zeros(T, E, device=DEVICE) for _ in range(4))
    F_ret, F_len, F_crash = (torch.full((T, E), float("nan"), device=DEVICE) for _ in range(3))

    t0 = time.time(); t_log = t0; steps_log = env_steps
    print(f"training: {E} cars x {T} steps = {E*T:,} decisions/update")
    try:
        while env_steps < TOTAL_STEPS:
            if MAX_HOURS and time.time() - t0 > MAX_HOURS * 3600:
                print("time budget reached"); break

            # ── collect rollout ──
            F_ret.fill_(float("nan")); F_len.fill_(float("nan")); F_crash.fill_(float("nan"))
            with torch.no_grad():
                for t in range(T):
                    logits, raw = net(obs)
                    dist = Categorical(logits=logits)
                    act = dist.sample()
                    B_obs[t], B_act[t], B_lp[t], B_val[t] = obs, act, dist.log_prob(act), rn.denorm(raw)
                    obs, obs_final, rew, crashed, trunc = cars.step(act)
                    done = crashed | trunc
                    ep_ret += rew; ep_len += 1
                    F_ret[t] = torch.where(done, ep_ret, F_ret[t])
                    F_len[t] = torch.where(done, ep_len, F_len[t])
                    F_crash[t] = torch.where(done, crashed.float(), F_crash[t])
                    ep_ret = torch.where(done, torch.zeros_like(ep_ret), ep_ret)
                    ep_len = torch.where(done, torch.zeros_like(ep_len), ep_len)
                    if trunc.any():   # time-limit cut-off is not a failure: bootstrap its value
                        vf = rn.denorm(net.critic(obs_final).squeeze(-1))
                        rew = rew + GAMMA * vf * trunc.float()
                    B_rew[t], B_done[t] = rew, done.float()
                last_val = rn.denorm(net.critic(obs).squeeze(-1))
            env_steps += E * T

            fin = ~torch.isnan(F_ret)
            for r, l, c in zip(F_ret[fin].tolist(), F_len[fin].tolist(), F_crash[fin].tolist()):
                recent_ret.append(r); recent_len.append(l); recent_crash.append(c); hist.append(r)
                episodes += 1

            # ── PPO update ──
            adv, ret = gae(B_rew, B_val, B_done, last_val)
            rn.update(ret)
            ret_n = rn.norm(ret).reshape(-1)
            adv = ((adv - adv.mean()) / (adv.std() + 1e-8)).reshape(-1)
            f_obs, f_act, f_lp = B_obs.reshape(-1, OBS_DIM), B_act.reshape(-1), B_lp.reshape(-1)
            N = f_obs.shape[0]
            pg_l, v_l, ent_l, kl_l = [], [], [], []
            for _ in range(EPOCHS):
                perm = torch.randperm(N, device=DEVICE)
                for s in range(0, N, MINIBATCH):
                    i = perm[s:s + MINIBATCH]
                    logits, raw = net(f_obs[i])
                    dist = Categorical(logits=logits)
                    logr = dist.log_prob(f_act[i]) - f_lp[i]
                    ratio = logr.exp()
                    pg = -torch.min(ratio * adv[i], ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS) * adv[i]).mean()
                    ent = dist.entropy().mean()
                    vl = 0.5 * ((raw - ret_n[i]) ** 2).mean()
                    opt.zero_grad(set_to_none=True)
                    (pg - ent_coef * ent + vl).backward()
                    nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD)
                    opt.step()
                    pg_l.append(pg.detach()); v_l.append(vl.detach()); ent_l.append(ent.detach())
                    kl_l.append(((ratio - 1) - logr).mean().detach())
                if torch.stack(kl_l[-(N // MINIBATCH or 1):]).mean() > 1.5 * TARGET_KL:
                    break
            ent_coef = max(ENT_MIN, ent_coef * ENT_DECAY)
            update += 1

            # ── log / eval / save ──
            if update % LOG_EVERY == 0 or update == 1:
                now = time.time()
                mean_ret = float(np.mean(recent_ret)) if recent_ret else float("nan")
                mean_len = float(np.mean(recent_len)) if recent_len else float("nan")
                crash_pct = 100 * float(np.mean(recent_crash)) if recent_crash else float("nan")
                print(f"upd {update:>6} | steps {env_steps/1e6:7.2f}M | {(env_steps-steps_log)/(now-t_log):7.0f} st/s | "
                      f"eps {episodes:>7} | ret {mean_ret:8.1f} | surv {mean_len/FPS:5.1f}s | crash {crash_pct:3.0f}% | "
                      f"ent {torch.stack(ent_l).mean().item():.3f} | kl {torch.stack(kl_l).mean().item():.4f} | "
                      f"{(now-t0)/60:6.1f} min")
                t_log, steps_log = now, env_steps
            if update % EVAL_EVERY == 0:
                score = run_eval(net, update, env_steps)
                t_log = time.time(); steps_log = env_steps
                if score > best:
                    best = score
                    snapshot("multi_best", npz=True)
                    print(f"     new multi_best.pt (validation score {best:.1f})")
            if update % SAVE_EVERY == 0:
                snapshot("multi_latest")
            if update % SNAP_EVERY == 0:
                snapshot(f"multi_snap_{update:05d}", npz=True)
    except KeyboardInterrupt:
        print("interrupted")
    finally:
        snapshot("multi_final", npz=True)
        print(f"saved {OUT_DIR}/multi_final.pt + multi_final.npz  ({episodes} episodes, {env_steps:,} decisions, {(time.time()-t0)/60:.1f} min)")
        try:
            run_eval(net, update, env_steps)
        except Exception as e:
            print("final eval skipped:", e)
    return net


net = train()
